# Notebook 03 — Data Curation
### Amazon LA Delivery Failure Prediction  ·  Correlation One — DANA, Week 12  ·  April 2026

---

This notebook covers the full data curation pipeline:

| Part | Topic |
|------|-------|
| **Part 1** | Data Sourcing |
| **Part 2** | Data Profiling — Structure, Content, Relationships |
| **Part 3** | Data Wrangling — Encoding, Feature Selection |
| **Part 4** | Final Schema Documentation |

---

> **Curation decision documented here:**  
> The raw files contain 3 extra columns generated from Barcelona open data  
> (`neighbourhood`, `accident_risk`, `traffic_congestion`). These are  
> **dropped** in this notebook and replaced by a geographically neutral  
> operational feature set aligned with LA last-mile logistics.

## Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas numpy matplotlib seaborn scikit-learn openpyxl
    print('Dependencies installed.')
else:
    print('Local environment detected.')

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

# ── colour scheme ────────────────────────────────────────────────────────────
C_OK    = '#1976D2'
C_FAIL  = '#D32F2F'
C_NEU   = '#546E7A'
C_WARN  = '#F57C00'
C_PASS  = '#388E3C'

print('Imports OK ✓')

In [ ]:
# ── File paths ───────────────────────────────────────────────────────────────
if IN_COLAB:
    # Mount Drive or upload files manually:
    #   from google.colab import drive; drive.mount('/content/drive')
    DATA_DIR = Path('/content')
else:
    DATA_DIR = Path('../data')

TRAIN_PATH = DATA_DIR / 'packages_train.csv'
VAL_PATH   = DATA_DIR / 'packages_validation.csv'
TEST_PATH  = DATA_DIR / 'packages_test.csv'

# ── Final 13-column schema (spec) ────────────────────────────────────────────
# Columns dropped: neighbourhood, accident_risk, traffic_congestion (Barcelona)
SPEC_COLS = [
    'package_id', 'package_type', 'shift', 'carrier',
    'route_distance_km', 'packages_in_route',
    'double_scan', 'locker_issue', 'damaged_on_arrival',
    'weather_risk', 'cr_number_missing', 'days_in_fc',
    'delivery_failed'
]
FEATURE_COLS = [c for c in SPEC_COLS if c not in ('package_id', 'delivery_failed')]
TARGET_COL   = 'delivery_failed'

print('Paths configured.')

---
## Part 1 — Data Sourcing

The dataset is a synthetic recreation of Amazon LA last-mile operations, generated with `data/generate_data.py`. It captures operational patterns across three splits:

In [ ]:
# ── Load raw files ───────────────────────────────────────────────────────────
_raw_train = pd.read_csv(TRAIN_PATH)
_raw_test  = pd.read_csv(TEST_PATH)
_raw_val   = pd.read_csv(VAL_PATH)

print('Raw file shapes:')
print(f'  packages_train.csv      : {_raw_train.shape}')
print(f'  packages_validation.csv : {_raw_val.shape}')
print(f'  packages_test.csv       : {_raw_test.shape}')

print()
print('Columns in packages_train.csv:')
print(' ', _raw_train.columns.tolist())

# Extra columns that will be dropped during curation
BARCELONA_COLS = ['neighbourhood', 'accident_risk', 'traffic_congestion']
extra = [c for c in _raw_train.columns if c in BARCELONA_COLS]
print()
print(f'Extra Barcelona columns detected ({len(extra)}): {extra}')
print('  → These will be DROPPED in Part 3 (Data Wrangling).')

In [ ]:
# ── Data Sourcing summary table ──────────────────────────────────────────────

def _file_kb(path: Path) -> str:
    kb = path.stat().st_size / 1024
    return f'{kb:.0f} KB' if kb < 1024 else f'{kb/1024:.1f} MB'

def _fail_rate(df: pd.DataFrame) -> str:
    if TARGET_COL in df.columns:
        return f'{df[TARGET_COL].mean():.1%}'
    return 'N/A (no label)'

sourcing = pd.DataFrame([
    {'File': 'packages_train.csv',
     'Records': f'{len(_raw_train):,}',
     'Raw cols': _raw_train.shape[1],
     'Curated cols': len(SPEC_COLS),
     'Failure Rate': _fail_rate(_raw_train),
     'Size': _file_kb(TRAIN_PATH),
     'Role': 'Primary analysis / model training'},
    {'File': 'packages_validation.csv',
     'Records': f'{len(_raw_val):,}',
     'Raw cols': _raw_val.shape[1],
     'Curated cols': _raw_val.shape[1],
     'Failure Rate': _fail_rate(_raw_val),
     'Size': _file_kb(VAL_PATH),
     'Role': 'Inference / holdout (Amazon LMRC real data)'},
    {'File': 'packages_test.csv',
     'Records': f'{len(_raw_test):,}',
     'Raw cols': _raw_test.shape[1],
     'Curated cols': len(SPEC_COLS),
     'Failure Rate': _fail_rate(_raw_test),
     'Size': _file_kb(TEST_PATH),
     'Role': 'Final evaluation'},
])

print('=' * 85)
print('  DATA SOURCING SUMMARY')
print('=' * 85)
print(sourcing.to_string(index=False))
print(f'\n  Total records (train + test) : {len(_raw_train)+len(_raw_test):,}')
print(f'  Primary analysis             : packages_train.csv ({len(_raw_train):,} records)')
print(f'  Generation script            : data/generate_data.py')
print(f'  Validation source            : Amazon LMRC 2021 (real, multi-city US)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))

files   = ['Train\n(5,000)', 'Validation\n(3,559)', 'Test\n(1,000)']
counts  = [len(_raw_train), len(_raw_val), len(_raw_test)]
c_bars  = [C_OK, C_WARN, C_NEU]

bars = ax.bar(files, counts, color=c_bars, alpha=0.85, width=0.45)
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{n:,}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_ylabel('Records')
ax.set_title('Dataset Splits — Record Counts', fontweight='bold')
p_synth = mpatches.Patch(color=C_OK,   label='Synthetic (generated)')
p_real  = mpatches.Patch(color=C_WARN, label='Real (Amazon LMRC 2021)')
ax.legend(handles=[p_synth, p_real])
plt.tight_layout()
plt.show()

---
## Part 2 — Data Profiling

All profiling is performed on **packages_train.csv** (5,000 records) after selecting the 13-column curated schema.

In [ ]:
# ── Apply curated schema — drop Barcelona columns ────────────────────────────
df = _raw_train[SPEC_COLS].copy()

print(f'Schema applied. Shape: {df.shape}')
print(f'Dropped : {[c for c in _raw_train.columns if c not in SPEC_COLS]}')
print()
df.head(3)

### 2.1 Structure Discovery — dtypes, nulls, ranges, cardinality

In [ ]:
def _mode_or_dash(series: pd.Series):
    if series.dtype == object:
        return series.mode()[0]
    return '—'

rows = []
for col in df.columns:
    s = df[col]
    rows.append({
        'Column'      : col,
        'Dtype'       : str(s.dtype),
        'Nulls'       : int(s.isnull().sum()),
        'Null %'      : f'{s.isnull().mean()*100:.1f}%',
        'Unique'      : s.nunique(),
        'Min'         : str(s.min()) if s.dtype != object else sorted(s.unique())[0],
        'Max'         : str(s.max()) if s.dtype != object else sorted(s.unique())[-1],
        'Mean / Mode' : f'{s.mean():.3f}' if s.dtype != object else _mode_or_dash(s),
    })

structure_df = pd.DataFrame(rows)
print('STRUCTURE DISCOVERY — packages_train.csv (curated 13 columns)')
print('=' * 95)
print(structure_df.to_string(index=False))

In [ ]:
total_nulls  = df.isnull().sum().sum()
total_cells  = df.shape[0] * df.shape[1]
completeness = 1 - total_nulls / total_cells

status = f'✓  PASS' if total_nulls == 0 else f'✗  FAIL — {total_nulls} nulls found'

print(f'Missing Values Audit')
print(f'  Total null values : {total_nulls}')
print(f'  Total cells       : {total_cells:,}')
print(f'  Completeness      : {completeness:.2%}')
print(f'  Data quality      : {status}')

In [ ]:
dtype_counts = df.dtypes.astype(str).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# dtype pie
axes[0].pie(dtype_counts.values, labels=dtype_counts.index,
            autopct='%1.0f%%', colors=[C_OK, C_NEU, C_WARN],
            startangle=90, textprops={'fontsize': 10})
axes[0].set_title('Column dtypes', fontweight='bold')

# cardinality bar
card = df.nunique().sort_values(ascending=False)
colors_card = [C_FAIL if v == 1 else C_WARN if v == 2 else C_OK for v in card.values]
axes[1].barh(card.index[::-1], card.values[::-1], color=colors_card[::-1], alpha=0.85)
axes[1].set_xlabel('Unique values')
axes[1].set_title('Cardinality per column', fontweight='bold')
axes[1].axvline(2, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='Binary')
axes[1].legend(fontsize=8)

plt.suptitle('Structure Discovery', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 2.2 Content Discovery — Distributions, outliers, class balance

In [ ]:
cat_cols = ['package_type', 'shift', 'carrier', 'weather_risk']

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts()
    bars = ax.bar(vc.index, vc.values / len(df) * 100,
                  color=C_NEU, alpha=0.85, edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('% of records')
    ax.tick_params(axis='x', rotation=30)
    for bar, v in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.4,
                f'{v/len(df)*100:.1f}%',
                ha='center', va='bottom', fontsize=8)

plt.suptitle('Categorical Distributions — Train (n=5,000)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print('Categorical value counts — Train')
print()
for col in cat_cols:
    vc = df[col].value_counts()
    print(f'  {col}:')
    for k, v in vc.items():
        print(f'    {k:<18} {v:>5}  ({v/len(df)*100:.1f}%)')
    print()

In [ ]:
num_cols = ['route_distance_km', 'packages_in_route', 'days_in_fc']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(num_cols):
    s = df[col]

    # histogram
    ax_h = axes[0][i]
    ax_h.hist(s, bins=30, color=C_OK, alpha=0.8, edgecolor='white')
    ax_h.axvline(s.mean(),   color=C_FAIL, linestyle='--', lw=1.5,
                 label=f'Mean {s.mean():.1f}')
    ax_h.axvline(s.median(), color=C_WARN, linestyle='--', lw=1.5,
                 label=f'Median {s.median():.1f}')
    ax_h.set_title(col, fontweight='bold')
    ax_h.set_ylabel('Frequency')
    ax_h.legend(fontsize=8)

    # box split by failed/not-failed
    ax_b = axes[1][i]
    g0 = df[df[TARGET_COL] == 0][col]
    g1 = df[df[TARGET_COL] == 1][col]
    bp = ax_b.boxplot([g0, g1], patch_artist=True,
                      labels=['Delivered', 'Failed'],
                      medianprops={'color': 'white', 'linewidth': 2})
    bp['boxes'][0].set_facecolor(C_OK)
    bp['boxes'][1].set_facecolor(C_FAIL)
    bp['boxes'][0].set_alpha(0.75)
    bp['boxes'][1].set_alpha(0.75)
    _, p = stats.mannwhitneyu(g0, g1, alternative='two-sided')
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax_b.set_title(f'{col} by outcome  (p={p:.4f} {sig})', fontweight='bold')
    ax_b.set_ylabel(col)

plt.suptitle('Numerical Features — Distributions & Outcome Split',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
print('Numerical Feature Statistics')
print(df[num_cols].describe().round(2).to_string())

In [ ]:
# ── Outlier audit: IQR method ────────────────────────────────────────────────
OPERATIONAL_RANGES = {
    'route_distance_km' : (2.0,  85.0),
    'packages_in_route' : (15,   120),
    'days_in_fc'        : (0,    12),
}

print('Outlier Audit')
print(f'{"Column":<22} {"IQR Outliers":>14} {"Below Op.Min":>14} {"Above Op.Max":>14} {"Decision"}')
print('-' * 80)

for col, (op_min, op_max) in OPERATIONAL_RANGES.items():
    s = df[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_out = int(((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum())
    below   = int((s < op_min).sum())
    above   = int((s > op_max).sum())
    decision = 'KEEP — within operational range' if (below + above) == 0 else 'REVIEW'
    print(f'{col:<22} {iqr_out:>14} {below:>14} {above:>14} {decision}')

print()
print('Outlier Treatment Decision:')
print('  RandomForest is non-parametric; tree splits are ordinal.')
print('  All values are within the defined operational range.')
print('  → No outlier removal applied.')

In [ ]:
# ── Binary flag positive rates ────────────────────────────────────────────────
binary_cols = ['double_scan', 'locker_issue', 'damaged_on_arrival',
               'cr_number_missing', TARGET_COL]

rates = df[binary_cols].mean() * 100

print('Binary Feature Positive Rates')
print(f'{"Flag":<22} {"Rate":>8}  {"Count":>7}')
print('-' * 45)
for col, r in rates.items():
    n = int(df[col].sum())
    tag = '  ← TARGET' if col == TARGET_COL else ''
    print(f'{col:<22} {r:>7.1f}%  {n:>7,}{tag}')

fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = [C_FAIL if c == TARGET_COL else C_NEU for c in binary_cols]
bars = ax.bar(binary_cols, rates.values, color=bar_colors, alpha=0.85)
for bar, v in zip(bars, rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')
ax.set_ylabel('Positive rate (%)')
ax.set_title('Binary Feature Positive Rates — Train (n=5,000)', fontweight='bold')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
vc = df[TARGET_COL].value_counts()
fail_rate = df[TARGET_COL].mean()

print('Class Balance — delivery_failed')
print(f'  Class 0 (Delivered) : {vc[0]:,}  ({1-fail_rate:.1%})')
print(f'  Class 1 (Failed)    : {vc[1]:,}  ({fail_rate:.1%})')
print(f'  Imbalance ratio     : {vc[0]/vc[1]:.1f}:1')
print(f'  Mitigation          : class_weight="balanced" in RandomForest')

fig, ax = plt.subplots(figsize=(5, 4))
ax.pie([vc[0], vc[1]],
       labels=[f'Delivered\n{1-fail_rate:.1%}', f'Failed\n{fail_rate:.1%}'],
       colors=[C_OK, C_FAIL], autopct='%1.1f%%',
       startangle=90, textprops={'fontsize': 11})
ax.set_title('Class Distribution — delivery_failed', fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Relationship Discovery — Correlations with target

In [ ]:
# Temporary label-encoding just for correlation analysis
df_enc = df.copy()
_le_tmp = LabelEncoder()
for col in ['package_type', 'shift', 'carrier', 'weather_risk']:
    df_enc[col] = _le_tmp.fit_transform(df_enc[col])
df_enc = df_enc.drop(columns=['package_id'])

corr_target = (df_enc.corr()[TARGET_COL]
               .drop(TARGET_COL)
               .sort_values(key=abs, ascending=False))

print('Pearson Correlations with delivery_failed (label-encoded categoricals)')
print(f'{"Feature":<25} {"Correlation":>12} {"Abs":>8} {"Strength"}')
print('-' * 65)
for feat, val in corr_target.items():
    strength = 'STRONG' if abs(val) > 0.15 else 'MODERATE' if abs(val) > 0.10 else 'WEAK'
    print(f'{feat:<25} {val:>12.4f} {abs(val):>8.4f}  {strength}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation bar chart (with target)
colors_corr = [C_FAIL if v > 0 else C_OK for v in corr_target.values]
axes[0].barh(corr_target.index[::-1], corr_target.values[::-1],
             color=colors_corr[::-1], alpha=0.85)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].axvline( 0.10, color='gray', linestyle='--', linewidth=1, alpha=0.6)
axes[0].axvline(-0.10, color='gray', linestyle='--', linewidth=1, alpha=0.6)
axes[0].set_xlabel('Pearson correlation with delivery_failed')
axes[0].set_title('Feature Correlations with Target', fontweight='bold')

# Full correlation heatmap
full_corr = df_enc.corr()
mask = np.triu(np.ones_like(full_corr, dtype=bool))
sns.heatmap(full_corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
            square=True, linewidths=0.4, ax=axes[1],
            annot_kws={'size': 7.5})
axes[1].set_title('Full Correlation Heatmap', fontweight='bold')

plt.suptitle('Relationship Discovery — Correlation Analysis',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Failure rate by each categorical ─────────────────────────────────────────
baseline = df[TARGET_COL].mean()

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, col in zip(axes, cat_cols):
    fr = (df.groupby(col)[TARGET_COL].mean() * 100).sort_values(ascending=False)
    bar_c = [C_FAIL if v/100 > baseline else C_OK for v in fr.values]
    bars = ax.bar(fr.index, fr.values, color=bar_c, alpha=0.85)
    ax.axhline(baseline * 100, color='black', linestyle='--', lw=1.2,
               label=f'Baseline {baseline:.1%}')
    ax.set_title(f'Failure rate by {col}', fontweight='bold')
    ax.set_ylabel('Failure %')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)
    for bar, v in zip(bars, fr.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=8)

plt.suptitle('Failure Rate by Categorical Feature — Train',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Part 3 — Data Wrangling

### 3.1 Categorical Encoding — LabelEncoder for RandomForest compatibility

**Rule:** Encoders are **fitted on TRAIN only** and then applied to val/test to prevent data leakage.

In [ ]:
CAT_COLS = ['package_type', 'shift', 'carrier', 'weather_risk']
CAT_ENC_MAP = {
    'package_type' : 'package_type_enc',
    'shift'        : 'shift_enc',
    'carrier'      : 'carrier_enc',
    'weather_risk'  : 'weather_enc',
}

# ── Fit encoders on train only ───────────────────────────────────────────────
encoders = {}
mapping_rows = []

for col in CAT_COLS:
    le = LabelEncoder()
    le.fit(df[col])   # ← train only
    encoders[col] = le
    mapping = {cls: i for i, cls in enumerate(le.classes_)}
    mapping_str = ', '.join(f'{k}={v}' for k, v in mapping.items())
    mapping_rows.append({
        'Source Column'  : col,
        'Encoded Column' : CAT_ENC_MAP[col],
        'Mapping'        : mapping_str,
    })

enc_table = pd.DataFrame(mapping_rows)
print('LabelEncoder Mapping Table (fitted on train only)')
print('=' * 75)
print(enc_table.to_string(index=False))

In [ ]:
def apply_encoders(raw_df: pd.DataFrame,
                   encoders: dict,
                   spec_cols: list,
                   has_target: bool = True) -> pd.DataFrame:
    """
    Select spec columns, drop Barcelona extras, apply pre-fitted encoders.
    Returns a curated DataFrame ready for modelling.
    """
    available = [c for c in spec_cols if c in raw_df.columns]
    out = raw_df[available].copy()

    for col, le in encoders.items():
        if col in out.columns:
            out[CAT_ENC_MAP[col]] = le.transform(out[col])

    return out


# Apply to all splits
df_train_cur = apply_encoders(_raw_train, encoders, SPEC_COLS, has_target=True)
df_test_cur  = apply_encoders(_raw_test,  encoders, SPEC_COLS, has_target=True)
df_val_cur   = apply_encoders(_raw_val,   encoders, list(_raw_val.columns), has_target=False)

print('Curated DataFrames:')
print(f'  Train      : {df_train_cur.shape}')
print(f'  Test       : {df_test_cur.shape}')
print(f'  Validation : {df_val_cur.shape}  (no delivery_failed label — Amazon LMRC)')
print()
print('Train — first 3 rows:')
df_train_cur.head(3)

In [ ]:
# ── Visualise: original vs encoded for each categorical ───────────────────────
fig, axes = plt.subplots(1, 4, figsize=(17, 4))

for ax, col in zip(axes, CAT_COLS):
    enc_col = CAT_ENC_MAP[col]
    le = encoders[col]
    vc = df_train_cur[col].value_counts()
    xpos = np.arange(len(vc))
    bars = ax.bar(xpos, vc.values, color=C_OK, alpha=0.85)
    ax.set_xticks(xpos)
    enc_labels = [f'{c}\n({le.transform([c])[0]})' for c in vc.index]
    ax.set_xticklabels(enc_labels, fontsize=8, rotation=30)
    ax.set_title(f'{col}  →  {enc_col}', fontweight='bold')
    ax.set_ylabel('Count')

plt.suptitle('LabelEncoder — Category → Integer Mapping (train)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 3.2 Final Feature Set

In [ ]:
MODEL_FEATURES = [
    'package_type_enc',
    'shift_enc',
    'carrier_enc',
    'route_distance_km',
    'packages_in_route',
    'double_scan',
    'locker_issue',
    'damaged_on_arrival',
    'weather_enc',
    'cr_number_missing',
    'days_in_fc',
]

X_train = df_train_cur[MODEL_FEATURES]
y_train = df_train_cur[TARGET_COL]

X_test  = df_test_cur[MODEL_FEATURES]
y_test  = df_test_cur[TARGET_COL]

print('Final Feature Matrix')
print(f'  MODEL_FEATURES       : {MODEL_FEATURES}')
print()
print(f'  X_train shape        : {X_train.shape}')
print(f'  y_train shape        : {y_train.shape}')
print(f'  X_test  shape        : {X_test.shape}')
print(f'  y_test  shape        : {y_test.shape}')
print()
print(f'  Class balance (train)')
for cls, n in y_train.value_counts().items():
    print(f'    {cls}: {n:,}  ({n/len(y_train):.3f})')

In [ ]:
print('Final Feature Matrix — Descriptive Statistics')
X_train.describe().round(3)

In [ ]:
# ── Feature × Target correlation in final feature space ──────────────────────
xy = X_train.copy()
xy[TARGET_COL] = y_train.values
corr_final = xy.corr()[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
bar_c = [C_FAIL if v > 0 else C_OK for v in corr_final.values]
ax.barh(corr_final.index, corr_final.values, color=bar_c, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.axvline( 0.10, color='gray', linestyle='--', linewidth=1, alpha=0.6, label='±0.10 threshold')
ax.axvline(-0.10, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel('Pearson correlation with delivery_failed')
ax.set_title('Final Model Features — Correlation with Target', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('\nTop features by |correlation|:')
for feat, val in corr_final.sort_values(key=abs, ascending=False).items():
    print(f'  {feat:<22} {val:+.4f}')

---
## Part 4 — Final Schema Documentation

In [ ]:
schema = [
    ('package_id',        'string',  '—',                  'ID',     'Unique package identifier (synthetic)'),
    ('package_type',      'string',  'package_type_enc',   'Feature','Package category — fragile/high_value/large/locker/standard'),
    ('shift',             'string',  'shift_enc',          'Feature','Delivery shift — morning/afternoon/night'),
    ('carrier',           'string',  'carrier_enc',        'Feature','Carrier — A=Amazon, B=SEUR, C=DHL, D=Correos'),
    ('route_distance_km', 'float64', 'as-is',              'Feature','Route length in km  (range: 2–85)'),
    ('packages_in_route', 'int64',   'as-is',              'Feature','Packages per driver route  (range: 15–120)'),
    ('double_scan',       'int64',   'as-is (binary)',     'Feature','Scan error flag  1=error detected'),
    ('locker_issue',      'int64',   'as-is (binary)',     'Feature','Locker unavailable at dispatch'),
    ('damaged_on_arrival','int64',   'as-is (binary)',     'Feature','Package damaged at FC receiving'),
    ('weather_risk',      'string',  'weather_enc',        'Feature','Environmental risk level — low/medium/high'),
    ('cr_number_missing', 'int64',   'as-is (binary)',     'Feature','Customer reference absent from record'),
    ('days_in_fc',        'int64',   'as-is',              'Feature','Days package waited in fulfillment center  (range: 0–12)'),
    ('delivery_failed',   'int64',   'as-is (binary)',     'TARGET', 'Delivery outcome  1=failed  0=delivered'),
]

schema_df = pd.DataFrame(schema,
    columns=['Column', 'Type', 'Encoded As', 'Role', 'Description'])

print('=' * 100)
print('  FINAL SCHEMA — Curated 13-Column Dataset')
print('=' * 100)
print(schema_df.to_string(index=False))

In [ ]:
# ── Schema at a glance — visual ───────────────────────────────────────────────
role_colors = {'ID': '#90A4AE', 'Feature': C_OK, 'TARGET': C_FAIL}
role_labels = {'ID': 'Identifier', 'Feature': 'Model Feature', 'TARGET': 'Target Variable'}

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis('off')

headers = ['Column', 'Type', 'Encoded As', 'Role', 'Description']
col_widths = [0.14, 0.07, 0.14, 0.07, 0.52]
x_positions = [sum(col_widths[:i]) + col_widths[i]/2 for i in range(len(col_widths))]

y_start = 0.95
row_h   = 0.065

# header row
for x, h in zip(x_positions, headers):
    ax.text(x, y_start, h, ha='center', va='center', fontsize=9, fontweight='bold',
            color='white',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#37474F', edgecolor='none'))

# data rows
for i, row in schema_df.iterrows():
    y = y_start - (i + 1) * row_h
    bg = '#ECEFF1' if i % 2 == 0 else 'white'
    role_bg = role_colors.get(row['Role'], C_NEU)

    ax.add_patch(plt.Rectangle((0, y - row_h*0.45), 1.0, row_h*0.9,
                                facecolor=bg, edgecolor='none', zorder=0))

    for j, (x, val) in enumerate(zip(x_positions, row.values)):
        color = 'white'
        bg_text = 'none'
        if j == 3:  # Role column
            bg_text = role_bg
        ax.text(x, y, str(val),
                ha='center' if j != 4 else 'left', va='center', fontsize=8,
                color='white' if bg_text != 'none' else '#263238',
                bbox=dict(boxstyle='round,pad=0.2',
                          facecolor=bg_text, edgecolor='none') if bg_text != 'none' else None)

# legend
patches = [mpatches.Patch(color=v, label=role_labels[k]) for k, v in role_colors.items()]
ax.legend(handles=patches, loc='lower right', bbox_to_anchor=(1.0, 0.0),
          fontsize=8, frameon=True)

ax.set_title('Final Schema — Curated 13-Column Dataset', fontweight='bold',
             fontsize=12, pad=8)
plt.tight_layout()
plt.show()

---
## Summary — Data Curation Findings

In [ ]:
fail_rate_train = df_train_cur[TARGET_COL].mean()
fail_rate_test  = df_test_cur[TARGET_COL].mean()

summary_rows = [
    ('Completeness',     '0% missing values in train/test',     'PASS ✓'),
    ('Consistency',      'All values within operational ranges', 'PASS ✓'),
    ('Outliers',         'No removal — RF robust to range extremes', 'PASS ✓'),
    ('Barcelona columns',f'Dropped 3 cols: neighbourhood, accident_risk, traffic_congestion', 'DONE ✓'),
    ('Encoding',         'LabelEncoder fitted on train only',    'PASS ✓'),
    ('Class balance',    f'Train: {1-fail_rate_train:.1%} / {fail_rate_train:.1%} → class_weight="balanced"', 'NOTE ⚠'),
    ('Leakage check',    'Encoders never exposed to val/test before fit', 'PASS ✓'),
    ('Feature count',    f'{len(MODEL_FEATURES)} model features + 1 target', 'DONE ✓'),
]

summary_df = pd.DataFrame(summary_rows, columns=['Check', 'Detail', 'Status'])

print('=' * 80)
print('  DATA CURATION SUMMARY')
print('=' * 80)
print(summary_df.to_string(index=False))
print()
print(f'  Curated train  : {X_train.shape[0]:,} rows × {X_train.shape[1]} features')
print(f'  Curated test   : {X_test.shape[0]:,} rows  × {X_test.shape[1]} features')
print(f'  Failure rate (train) : {fail_rate_train:.1%}')
print(f'  Failure rate (test)  : {fail_rate_test:.1%}')
print()
print('  Data is clean, encoded, and ready for model training.')

In [ ]:
# ── Curation pipeline diagram ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 3))
ax.axis('off')

steps = [
    ('Raw Data\n(16 cols)', '#CFD8DC'),
    ('Drop Barcelona\nCols  (−3)', '#FFCDD2'),
    ('Null Audit\n0 nulls', '#C8E6C9'),
    ('Outlier Check\nNo removal', '#C8E6C9'),
    ('LabelEncoder\n(train only)', '#BBDEFB'),
    ('Final Feature\nMatrix  11×5000', '#DCEDC8'),
]

n = len(steps)
xs = np.linspace(0.08, 0.92, n)

for i, (label, color) in enumerate(steps):
    ax.add_patch(plt.FancyBboxPatch((xs[i]-0.07, 0.2), 0.14, 0.55,
                                     boxstyle='round,pad=0.02',
                                     facecolor=color, edgecolor='#546E7A',
                                     linewidth=1.2))
    ax.text(xs[i], 0.48, label, ha='center', va='center', fontsize=9,
            fontweight='bold', color='#263238')
    if i < n - 1:
        ax.annotate('', xy=(xs[i+1]-0.075, 0.475), xytext=(xs[i]+0.075, 0.475),
                    arrowprops=dict(arrowstyle='->', color='#546E7A', lw=1.5))

ax.set_title('Data Curation Pipeline', fontsize=13, fontweight='bold', pad=6)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()